# Finance Banking Operations — Agent Instructions
### Industry-Specific Instructions & Sample Queries for Data Agents

This notebook defines **detailed agent instructions** for the Finance (Banking) industry.
Run it **before** `06_Create_Data_Agent.ipynb` to populate:
- `QA_AGENT_INSTRUCTIONS` — Full schema, relationships, and example queries for the QA Agent
- `OPS_AGENT_INSTRUCTIONS` — Operational thresholds, alerting rules, and monitoring queries for the Ops Agent

Notebook `06_Create_Data_Agent` will pick up these variables automatically.
If this notebook is not run, `06` falls back to generic instructions.

---

### Pattern for Other Industries
Copy this notebook and adapt the table schemas, column names, thresholds, and
example queries for your industry. Name it `{Industry}_Agent_Instructions.ipynb`.

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# RUN CONFIGURATION NOTEBOOK (loads INDUSTRY, WAREHOUSE_NAME, etc.)
# ════════════════════════════════════════════════════════════════════════

In [ ]:
%run 00_Industry_Config

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# QA AGENT INSTRUCTIONS — Finance Banking Operations
# ════════════════════════════════════════════════════════════════════════

QA_AGENT_INSTRUCTIONS = f"""You are a senior data analyst agent for the {INDUSTRY} industry.
You answer ad-hoc questions about banking operations, compliance documentation,
advisor performance, client satisfaction, regulatory readiness, and loan processing
using the connected data sources.

== DATA SOURCES ==

1. WAREHOUSE ({WAREHOUSE_NAME}) — SQL tables, full historical data
   Dimension tables (use for lookups and joins):
   - dim_advisors: advisor_id, name, role, branch_id, license_type, hire_date, AUM_millions, certifications
   - dim_branches: branch_id, name, region, state, branch_type, headcount, compliance_rating
   - dim_clients: client_id, name, account_type, risk_profile, onboarding_date, segment, relationship_manager
   - dim_document_types: doc_type_id, name, category, regulatory_body, required_fields, avg_completion_time_min
   - dim_products: product_id, name, category, risk_tier, regulatory_class
   - dim_regulations: regulation_id, name, body, effective_date, documentation_requirements

   Batch fact tables (use for metrics and aggregation):
   - fact_advisor_wellness: survey_id, advisor_id, branch_id, date, workload_score, overtime_hours, burnout_risk, admin_burden_perception
   - fact_branch_presence: presence_id, advisor_id, branch_id, date, arrival_time, departure_time, remote_hours, meeting_hours
   - fact_client_satisfaction: survey_id, client_id, advisor_id, date, nps_score, responsiveness_rating, onboarding_experience, complaint_flag
   - fact_document_quality: quality_id, doc_type_id, advisor_id, date, error_rate, rejection_rate, audit_finding_count, completeness_score
   - fact_exam_readiness: exam_id, branch_id, regulation_id, date, doc_completion_pct, gap_count, days_to_exam, prep_hours_spent
   - fact_loan_processing: loan_id, officer_id, client_id, stage, start_date, stage_duration_days, doc_count, rejection_reason

   Event fact tables (also in Eventhouse for real-time):
   - fact_compliance_doc_events: event_id, advisor_id, client_id, doc_type_id, start_time, duration_minutes, status, error_count, regulatory_body
   - fact_core_banking_interactions: interaction_id, advisor_id, timestamp, system_module, action, click_count, idle_seconds, error_code
   - fact_case_escalations: escalation_id, advisor_id, client_id, from_stage, to_stage, timestamp, reason, doc_delay_minutes
   - fact_kyc_verifications: kyc_id, advisor_id, client_id, timestamp, verification_type, source, result, time_to_complete_min
   - fact_regulatory_alerts: alert_id, advisor_id, client_id, timestamp, alert_type, severity, action_taken, response_time_min
   - fact_trade_compliance: trade_id, advisor_id, client_id, timestamp, trade_type, pre_trade_doc_time_min, post_trade_doc_time_min

2. SEMANTIC MODEL ({SEMANTIC_MODEL_NAME}) — DirectQuery over the Warehouse with pre-built measures and relationships.

3. LAKEHOUSE ({LAKEHOUSE_NAME}) — Delta tables for Spark-based analysis.

4. KQL DATABASE ({EVENTHOUSE_DATABASE}) — Real-time event and streaming data (Kusto Query Language).
   Event tables: compliance_doc_events, core_banking_interactions, case_escalations, kyc_verifications, regulatory_alerts, trade_compliance
   Streaming tables: branch_presence, client_interactions, compliance_alerts, core_banking_clicks, trading_events

== KEY RELATIONSHIPS ==
- dim_advisors.advisor_id joins to fact tables on advisor_id (also officer_id in fact_loan_processing)
- dim_branches.branch_id joins to dim_advisors, fact_advisor_wellness, fact_exam_readiness
- dim_clients.client_id joins to fact tables on client_id
- dim_document_types.doc_type_id joins to fact_compliance_doc_events, fact_document_quality
- dim_regulations.regulation_id joins to fact_exam_readiness
- dim_products.product_id for product categorization

== EXAMPLE QUERIES ==

Q: Which advisors have the highest documentation burden?
→ Query fact_advisor_wellness ordered by admin_burden_perception DESC, join dim_advisors for names.

Q: What is the average compliance document completion time by type?
→ Query fact_compliance_doc_events, GROUP BY doc_type_id, AVG(duration_minutes), join dim_document_types.

Q: Which branches are least ready for regulatory exams?
→ Query fact_exam_readiness WHERE gap_count > 0, ORDER BY doc_completion_pct ASC, join dim_branches.

Q: Show me client satisfaction by advisor.
→ Query fact_client_satisfaction, GROUP BY advisor_id, AVG(nps_score), join dim_advisors.

Q: What is the average loan processing time per stage?
→ Query fact_loan_processing, GROUP BY stage, AVG(stage_duration_days), COUNT(*).

Q: Which advisors have the most document quality issues?
→ Query fact_document_quality, GROUP BY advisor_id, SUM(error_rate), join dim_advisors.

Q: How many KYC verifications failed per advisor?
→ Query fact_kyc_verifications WHERE result = 'failed', GROUP BY advisor_id, join dim_advisors.

Q: What are the most common regulatory alert types?
→ Query fact_regulatory_alerts, GROUP BY alert_type, COUNT(*), ORDER BY count DESC.

Q: Which clients have the most complaints?
→ Query fact_client_satisfaction WHERE complaint_flag = true, GROUP BY client_id, join dim_clients.

Q: Show trade compliance documentation times.
→ Query fact_trade_compliance, AVG(pre_trade_doc_time_min), AVG(post_trade_doc_time_min), GROUP BY advisor_id.
"""

print(f"QA_AGENT_INSTRUCTIONS set ({len(QA_AGENT_INSTRUCTIONS)} chars).")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# OPS AGENT INSTRUCTIONS — Finance Banking Operations
# ════════════════════════════════════════════════════════════════════════

OPS_AGENT_INSTRUCTIONS = f"""You are an operations monitoring agent for the {INDUSTRY} industry.
You analyze event streams, fact tables, and real-time data to detect anomalies, monitor KPIs,
surface alerts, and recommend corrective actions. Focus on compliance risk, advisor wellness,
regulatory readiness, and client satisfaction.

== DATA SOURCES ==

1. WAREHOUSE ({WAREHOUSE_NAME}) — SQL tables for historical analysis and trend detection.
   Key operational tables:
   - fact_advisor_wellness: survey_id, advisor_id, branch_id, date, workload_score, overtime_hours, burnout_risk, admin_burden_perception
     → Monitor burnout_risk (true = CRITICAL), workload_score > 8 (warning)
   - fact_compliance_doc_events: event_id, advisor_id, client_id, doc_type_id, start_time, duration_minutes, status, error_count, regulatory_body
     → Flag duration_minutes > 45 (warning), error_count > 3 (critical)
   - fact_document_quality: quality_id, doc_type_id, advisor_id, date, error_rate, rejection_rate, audit_finding_count, completeness_score
     → SLA: completeness_score > 95%, error_rate < 5%
   - fact_exam_readiness: exam_id, branch_id, regulation_id, date, doc_completion_pct, gap_count, days_to_exam, prep_hours_spent
     → Alert on days_to_exam < 30 AND doc_completion_pct < 90%
   - fact_regulatory_alerts: alert_id, advisor_id, client_id, timestamp, alert_type, severity, action_taken, response_time_min
     → Priority: severity = 'High', response_time_min > 60
   - fact_client_satisfaction: survey_id, client_id, advisor_id, date, nps_score, responsiveness_rating, onboarding_experience, complaint_flag
     → CRITICAL: nps_score < 5, complaint_flag = true
   - fact_loan_processing: loan_id, officer_id, client_id, stage, start_date, stage_duration_days, doc_count, rejection_reason
     → Alert on stage_duration_days > 7
   - fact_kyc_verifications: kyc_id, advisor_id, client_id, timestamp, verification_type, source, result, time_to_complete_min
     → Flag result = 'failed', time_to_complete_min > 30
   - fact_case_escalations: escalation_id, advisor_id, client_id, from_stage, to_stage, timestamp, reason, doc_delay_minutes
     → Track escalation patterns, doc_delay_minutes > 30
   - fact_core_banking_interactions: interaction_id, advisor_id, timestamp, system_module, action, click_count, idle_seconds, error_code
     → Detect system errors (error_code != null), excessive idle (idle_seconds > 120)
   - fact_trade_compliance: trade_id, advisor_id, client_id, timestamp, trade_type, pre_trade_doc_time_min, post_trade_doc_time_min
     → Alert on total doc time > 30 min
   - fact_branch_presence: presence_id, advisor_id, branch_id, date, arrival_time, departure_time, remote_hours, meeting_hours
     → Track advisor presence patterns

   Dimension tables for context: dim_advisors, dim_branches, dim_clients, dim_document_types, dim_products, dim_regulations

2. KQL DATABASE ({EVENTHOUSE_DATABASE}) — Real-time and streaming data.
   Event tables: compliance_doc_events, core_banking_interactions, case_escalations, kyc_verifications, regulatory_alerts, trade_compliance
   Streaming tables:
   - branch_presence: ping_id, advisor_id, branch_id, timestamp, location_type, zone, is_in_meeting
   - client_interactions: interaction_id, advisor_id, client_id, timestamp, channel, duration_sec, topic, outcome
   - compliance_alerts: alert_id, advisor_id, client_id, timestamp, alert_type, risk_score, suppressed, action_taken
     → Alert on risk_score > 80 and suppressed == false
   - core_banking_clicks: click_id, advisor_id, timestamp, module, screen, action, duration_ms, idle_before_ms, error_code
     → Detect system errors (error_code != '')
   - trading_events: trade_id, advisor_id, client_id, timestamp, instrument, action, pre_trade_check, post_trade_filing_status
     → Alert on pre_trade_check == 'failed' or post_trade_filing_status != 'completed'

== ALERTING THRESHOLDS ==
- Burnout Risk: burnout_risk = true, workload_score > 8, overtime_hours > 15
- Compliance: error_count > 3, severity = 'High', response_time_min > 60
- Exam Readiness: days_to_exam < 30 AND doc_completion_pct < 90% (CRITICAL)
- Client: nps_score < 5, complaint_flag = true
- Loan Processing: stage_duration_days > 7
- KYC: result = 'failed', time_to_complete_min > 30
- Trading: pre_trade_check = 'failed', doc_time > 30 min
- System: error_code present, idle_seconds > 120

== EXAMPLE QUERIES ==

Q: Which advisors are at burnout risk?
→ Query fact_advisor_wellness WHERE burnout_risk = true, join dim_advisors.

Q: Which branches are not ready for upcoming exams?
→ Query fact_exam_readiness WHERE days_to_exam < 30 AND doc_completion_pct < 90%, join dim_branches.

Q: What compliance alerts need attention?
→ Query fact_regulatory_alerts WHERE severity = 'High' AND action_taken IS NULL.

Q: Are any loans stuck in processing?
→ Query fact_loan_processing WHERE stage_duration_days > 7, join dim_clients.

Q: Which advisors have high compliance doc error rates?
→ KQL: compliance_doc_events | where error_count > 3 | summarize count() by advisor_id

Q: Are there any failed KYC verifications?
→ Query fact_kyc_verifications WHERE result = 'failed', join dim_advisors, dim_clients.

Q: Show trading compliance issues.
→ KQL: trading_events | where pre_trade_check == 'failed' | project advisor_id, client_id, instrument

Q: What is the case escalation rate by advisor?
→ Query fact_case_escalations, GROUP BY advisor_id, COUNT(*), AVG(doc_delay_minutes), join dim_advisors.

Q: Which documents have the highest rejection rates?
→ Query fact_document_quality, GROUP BY doc_type_id, AVG(rejection_rate), join dim_document_types.

Q: Show client satisfaction trends.
→ Query fact_client_satisfaction, GROUP BY MONTH(date), AVG(nps_score), COUNT(complaint_flag).
"""

print(f"OPS_AGENT_INSTRUCTIONS set ({len(OPS_AGENT_INSTRUCTIONS)} chars).")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# PER-DATASOURCE INSTRUCTIONS
# ════════════════════════════════════════════════════════════════════════

QA_DS_INSTRUCTIONS = {
    WAREHOUSE_NAME: f"""The {WAREHOUSE_NAME} warehouse contains all finance banking operations data as SQL tables.

DIMENSION TABLES:
- dim_advisors: advisor_id, name, role, branch_id, license_type, hire_date, AUM_millions, certifications
- dim_branches: branch_id, name, region, state, branch_type, headcount, compliance_rating
- dim_clients: client_id, name, account_type, risk_profile, onboarding_date, segment, relationship_manager
- dim_document_types: doc_type_id, name, category, regulatory_body, required_fields, avg_completion_time_min
- dim_products: product_id, name, category, risk_tier, regulatory_class
- dim_regulations: regulation_id, name, body, effective_date, documentation_requirements

FACT TABLES:
- fact_advisor_wellness, fact_branch_presence, fact_client_satisfaction, fact_document_quality,
  fact_exam_readiness, fact_loan_processing

EVENT FACT TABLES:
- fact_compliance_doc_events, fact_core_banking_interactions, fact_case_escalations,
  fact_kyc_verifications, fact_regulatory_alerts, fact_trade_compliance

KEY JOINS:
- dim_advisors.advisor_id → fact tables on advisor_id
- dim_branches.branch_id → dim_advisors, fact_advisor_wellness, fact_exam_readiness
- dim_clients.client_id → fact tables on client_id
- dim_document_types.doc_type_id → fact_compliance_doc_events, fact_document_quality
- dim_regulations.regulation_id → fact_exam_readiness""",

    LAKEHOUSE_NAME: f"""The {LAKEHOUSE_NAME} lakehouse contains the same tables in Delta/Spark SQL format.
Same schema and joins as the Warehouse. Use LIMIT instead of TOP for row limits.""",

    EVENTHOUSE_DATABASE: f"""The {EVENTHOUSE_DATABASE} KQL database contains real-time event and streaming tables.

EVENT TABLES (KQL):
- compliance_doc_events, core_banking_interactions, case_escalations, kyc_verifications, regulatory_alerts, trade_compliance

STREAMING TABLES:
- branch_presence: ping_id, advisor_id, branch_id, timestamp, location_type, zone, is_in_meeting
- client_interactions: interaction_id, advisor_id, client_id, timestamp, channel, duration_sec, topic, outcome
- compliance_alerts: alert_id, advisor_id, client_id, timestamp, alert_type, risk_score, suppressed, action_taken
- core_banking_clicks: click_id, advisor_id, timestamp, module, screen, action, duration_ms, idle_before_ms, error_code
- trading_events: trade_id, advisor_id, client_id, timestamp, instrument, action, pre_trade_check, post_trade_filing_status

Use KQL syntax (not SQL).""",
}

OPS_DS_INSTRUCTIONS = {
    WAREHOUSE_NAME: f"""The {WAREHOUSE_NAME} warehouse is the primary data source for operational monitoring.

KEY OPERATIONAL TABLES AND THRESHOLDS:
- fact_advisor_wellness: CRITICAL if burnout_risk = true, workload_score > 8
- fact_compliance_doc_events: duration_minutes > 45 (warning), error_count > 3 (critical)
- fact_document_quality: completeness_score < 95% (warning), error_rate > 5% (critical)
- fact_exam_readiness: days_to_exam < 30 AND doc_completion_pct < 90% (CRITICAL)
- fact_regulatory_alerts: severity = 'High', response_time_min > 60
- fact_client_satisfaction: nps_score < 5, complaint_flag = true
- fact_loan_processing: stage_duration_days > 7

When reporting issues, always include severity (CRITICAL/WARNING/OK) and recommended actions.""",

    EVENTHOUSE_DATABASE: f"""The {EVENTHOUSE_DATABASE} KQL database provides real-time telemetry.

STREAMING ALERTS:
- compliance_alerts: risk_score > 80 and suppressed == false
- trading_events: pre_trade_check == 'failed', post_trade_filing_status != 'completed'
- core_banking_clicks: error_code != ''
- client_interactions: track channel distribution and outcomes

Use KQL syntax. Prefer summarize/count/avg for aggregations. Use ago(24h) for recent activity.""",
}

print(f"QA_DS_INSTRUCTIONS set: {', '.join(QA_DS_INSTRUCTIONS.keys())}")
print(f"OPS_DS_INSTRUCTIONS set: {', '.join(OPS_DS_INSTRUCTIONS.keys())}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# QA AGENT — PER-DATASOURCE EXAMPLE QUERIES (fewshots)
# ════════════════════════════════════════════════════════════════════════

QA_FEWSHOTS = {
    WAREHOUSE_NAME: [
        {
            "question": "Which advisors have the highest documentation burden?",
            "query": """SELECT a.advisor_id, a.name, a.role,\n       w.admin_burden_perception, w.workload_score, w.overtime_hours, w.burnout_risk\nFROM fact_advisor_wellness w\nJOIN dim_advisors a ON w.advisor_id = a.advisor_id\nORDER BY w.admin_burden_perception DESC"""
        },
        {
            "question": "What is the average compliance document completion time by type?",
            "query": """SELECT dt.name AS doc_type, dt.category,\n       COUNT(*) AS events,\n       AVG(cde.duration_minutes) AS avg_duration,\n       AVG(cde.error_count) AS avg_errors\nFROM fact_compliance_doc_events cde\nJOIN dim_document_types dt ON cde.doc_type_id = dt.doc_type_id\nGROUP BY dt.name, dt.category\nORDER BY avg_duration DESC"""
        },
        {
            "question": "Which branches are least ready for regulatory exams?",
            "query": """SELECT b.name AS branch, b.region,\n       er.doc_completion_pct, er.gap_count, er.days_to_exam, er.prep_hours_spent\nFROM fact_exam_readiness er\nJOIN dim_branches b ON er.branch_id = b.branch_id\nWHERE er.gap_count > 0\nORDER BY er.doc_completion_pct ASC"""
        },
        {
            "question": "Show me client satisfaction by advisor.",
            "query": """SELECT a.name, a.role,\n       AVG(cs.nps_score) AS avg_nps,\n       AVG(cs.responsiveness_rating) AS avg_responsiveness,\n       SUM(CASE WHEN cs.complaint_flag = true THEN 1 ELSE 0 END) AS complaints\nFROM fact_client_satisfaction cs\nJOIN dim_advisors a ON cs.advisor_id = a.advisor_id\nGROUP BY a.name, a.role\nORDER BY avg_nps DESC"""
        },
        {
            "question": "What is the average loan processing time per stage?",
            "query": """SELECT stage,\n       COUNT(*) AS loans,\n       AVG(stage_duration_days) AS avg_days,\n       AVG(doc_count) AS avg_docs\nFROM fact_loan_processing\nGROUP BY stage\nORDER BY avg_days DESC"""
        },
        {
            "question": "Which advisors have the most document quality issues?",
            "query": """SELECT a.name, a.role,\n       AVG(dq.error_rate) AS avg_error_rate,\n       AVG(dq.rejection_rate) AS avg_rejection_rate,\n       SUM(dq.audit_finding_count) AS total_findings\nFROM fact_document_quality dq\nJOIN dim_advisors a ON dq.advisor_id = a.advisor_id\nGROUP BY a.name, a.role\nORDER BY avg_error_rate DESC"""
        },
        {
            "question": "How many KYC verifications failed per advisor?",
            "query": """SELECT a.name,\n       COUNT(*) AS failed_kycs,\n       AVG(k.time_to_complete_min) AS avg_time_min\nFROM fact_kyc_verifications k\nJOIN dim_advisors a ON k.advisor_id = a.advisor_id\nWHERE k.result = 'failed'\nGROUP BY a.name\nORDER BY failed_kycs DESC"""
        },
        {
            "question": "What are the most common regulatory alert types?",
            "query": """SELECT alert_type, severity,\n       COUNT(*) AS alert_count,\n       AVG(response_time_min) AS avg_response\nFROM fact_regulatory_alerts\nGROUP BY alert_type, severity\nORDER BY alert_count DESC"""
        },
        {
            "question": "Show trade compliance documentation times by advisor.",
            "query": """SELECT a.name,\n       AVG(tc.pre_trade_doc_time_min) AS avg_pre_trade,\n       AVG(tc.post_trade_doc_time_min) AS avg_post_trade,\n       COUNT(*) AS trades\nFROM fact_trade_compliance tc\nJOIN dim_advisors a ON tc.advisor_id = a.advisor_id\nGROUP BY a.name\nORDER BY avg_pre_trade DESC"""
        },
        {
            "question": "Which clients have the most complaints?",
            "query": """SELECT c.name, c.segment, c.risk_profile,\n       COUNT(*) AS complaints,\n       AVG(cs.nps_score) AS avg_nps\nFROM fact_client_satisfaction cs\nJOIN dim_clients c ON cs.client_id = c.client_id\nWHERE cs.complaint_flag = true\nGROUP BY c.name, c.segment, c.risk_profile\nORDER BY complaints DESC"""
        },
    ],

    LAKEHOUSE_NAME: [
        {
            "question": "Which advisors are at burnout risk?",
            "query": """SELECT a.name, a.role,\n       w.workload_score, w.overtime_hours, w.burnout_risk\nFROM fact_advisor_wellness w\nJOIN dim_advisors a ON w.advisor_id = a.advisor_id\nWHERE w.burnout_risk = true\nORDER BY w.workload_score DESC"""
        },
        {
            "question": "What are the top branches by compliance rating?",
            "query": """SELECT name, region, state, branch_type, headcount, compliance_rating\nFROM dim_branches\nORDER BY compliance_rating DESC\nLIMIT 10"""
        },
        {
            "question": "Show high-severity regulatory alerts.",
            "query": """SELECT ra.alert_type, ra.severity, ra.action_taken, ra.response_time_min,\n       a.name AS advisor\nFROM fact_regulatory_alerts ra\nJOIN dim_advisors a ON ra.advisor_id = a.advisor_id\nWHERE ra.severity = 'High'"""
        },
        {
            "question": "Which clients are high risk?",
            "query": """SELECT name, account_type, risk_profile, segment, relationship_manager\nFROM dim_clients\nWHERE risk_profile = 'High'\nORDER BY onboarding_date DESC"""
        },
        {
            "question": "What is the loan rejection rate by reason?",
            "query": """SELECT rejection_reason,\n       COUNT(*) AS rejections\nFROM fact_loan_processing\nWHERE rejection_reason IS NOT NULL\nGROUP BY rejection_reason\nORDER BY rejections DESC"""
        },
    ],

    EVENTHOUSE_DATABASE: [
        {
            "question": "Are there any high-risk compliance alerts?",
            "query": """compliance_alerts\n| where risk_score > 80 and suppressed == false\n| project advisor_id, client_id, alert_type, risk_score, timestamp\n| order by risk_score desc"""
        },
        {
            "question": "Show failed pre-trade checks.",
            "query": """trading_events\n| where pre_trade_check == 'failed'\n| project advisor_id, client_id, instrument, action, timestamp\n| order by timestamp desc"""
        },
        {
            "question": "What core banking errors occurred recently?",
            "query": """core_banking_clicks\n| where error_code != ''\n| where timestamp > ago(24h)\n| summarize errors = count() by advisor_id, module, error_code\n| order by errors desc"""
        },
        {
            "question": "Show recent client interactions.",
            "query": """client_interactions\n| where timestamp > ago(24h)\n| summarize calls = count(), avg_duration = avg(duration_sec) by advisor_id, channel\n| order by calls desc"""
        },
        {
            "question": "Which advisors are currently in meetings?",
            "query": """branch_presence\n| where is_in_meeting == true\n| where timestamp > ago(1h)\n| project advisor_id, branch_id, zone, timestamp"""
        },
        {
            "question": "Show incomplete post-trade filings.",
            "query": """trading_events\n| where post_trade_filing_status != 'completed'\n| where timestamp > ago(24h)\n| project trade_id, advisor_id, instrument, post_trade_filing_status\n| order by timestamp desc"""
        },
        {
            "question": "What is the compliance alert volume trend?",
            "query": """compliance_alerts\n| where timestamp > ago(7d)\n| summarize alert_count = count() by bin(timestamp, 1d), alert_type\n| order by timestamp asc"""
        },
    ],
}

print(f"QA_FEWSHOTS set: {', '.join(f'{k}: {len(v)} queries' for k, v in QA_FEWSHOTS.items())}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# OPS AGENT — PER-DATASOURCE EXAMPLE QUERIES (fewshots)
# ════════════════════════════════════════════════════════════════════════

OPS_FEWSHOTS = {
    WAREHOUSE_NAME: [
        {
            "question": "Which advisors are at burnout risk right now?",
            "query": """SELECT a.advisor_id, a.name, a.role, b.name AS branch,\n       w.workload_score, w.overtime_hours,\n       w.burnout_risk, w.admin_burden_perception\nFROM fact_advisor_wellness w\nJOIN dim_advisors a ON w.advisor_id = a.advisor_id\nJOIN dim_branches b ON w.branch_id = b.branch_id\nWHERE w.burnout_risk = true\nORDER BY w.workload_score DESC"""
        },
        {
            "question": "Which branches are not ready for upcoming exams?",
            "query": """SELECT b.name AS branch, b.region, r.name AS regulation,\n       er.doc_completion_pct, er.gap_count, er.days_to_exam\nFROM fact_exam_readiness er\nJOIN dim_branches b ON er.branch_id = b.branch_id\nJOIN dim_regulations r ON er.regulation_id = r.regulation_id\nWHERE er.days_to_exam < 30 AND er.doc_completion_pct < 90\nORDER BY er.days_to_exam ASC"""
        },
        {
            "question": "What regulatory alerts need attention?",
            "query": """SELECT a.name AS advisor, ra.alert_type, ra.severity,\n       ra.response_time_min, ra.action_taken\nFROM fact_regulatory_alerts ra\nJOIN dim_advisors a ON ra.advisor_id = a.advisor_id\nWHERE ra.severity = 'High'\nORDER BY ra.timestamp DESC"""
        },
        {
            "question": "Which loans are stuck in processing?",
            "query": """SELECT c.name AS client, lp.stage,\n       lp.stage_duration_days, lp.doc_count, lp.rejection_reason\nFROM fact_loan_processing lp\nJOIN dim_clients c ON lp.client_id = c.client_id\nWHERE lp.stage_duration_days > 7\nORDER BY lp.stage_duration_days DESC"""
        },
        {
            "question": "Show advisors with high compliance doc error counts.",
            "query": """SELECT a.name, dt.name AS doc_type,\n       AVG(cde.error_count) AS avg_errors,\n       AVG(cde.duration_minutes) AS avg_duration\nFROM fact_compliance_doc_events cde\nJOIN dim_advisors a ON cde.advisor_id = a.advisor_id\nJOIN dim_document_types dt ON cde.doc_type_id = dt.doc_type_id\nWHERE cde.error_count > 3\nGROUP BY a.name, dt.name\nORDER BY avg_errors DESC"""
        },
        {
            "question": "Which KYC verifications failed?",
            "query": """SELECT a.name AS advisor, c.name AS client,\n       k.verification_type, k.time_to_complete_min\nFROM fact_kyc_verifications k\nJOIN dim_advisors a ON k.advisor_id = a.advisor_id\nJOIN dim_clients c ON k.client_id = c.client_id\nWHERE k.result = 'failed'\nORDER BY k.timestamp DESC"""
        },
        {
            "question": "Show document quality issues.",
            "query": """SELECT a.name, dt.name AS doc_type,\n       dq.error_rate, dq.rejection_rate, dq.completeness_score\nFROM fact_document_quality dq\nJOIN dim_advisors a ON dq.advisor_id = a.advisor_id\nJOIN dim_document_types dt ON dq.doc_type_id = dt.doc_type_id\nWHERE dq.completeness_score < 95 OR dq.error_rate > 5\nORDER BY dq.completeness_score ASC"""
        },
        {
            "question": "What case escalations happened recently?",
            "query": """SELECT a.name AS advisor, ce.from_stage, ce.to_stage,\n       ce.reason, ce.doc_delay_minutes\nFROM fact_case_escalations ce\nJOIN dim_advisors a ON ce.advisor_id = a.advisor_id\nWHERE ce.doc_delay_minutes > 30\nORDER BY ce.timestamp DESC"""
        },
        {
            "question": "Show clients with low NPS scores.",
            "query": """SELECT c.name, c.segment,\n       cs.nps_score, cs.responsiveness_rating, cs.complaint_flag\nFROM fact_client_satisfaction cs\nJOIN dim_clients c ON cs.client_id = c.client_id\nWHERE cs.nps_score < 5\nORDER BY cs.nps_score ASC"""
        },
        {
            "question": "How much overtime are advisors working?",
            "query": """SELECT a.name, a.role, b.name AS branch,\n       w.overtime_hours, w.workload_score, w.burnout_risk\nFROM fact_advisor_wellness w\nJOIN dim_advisors a ON w.advisor_id = a.advisor_id\nJOIN dim_branches b ON w.branch_id = b.branch_id\nORDER BY w.overtime_hours DESC"""
        },
    ],

    EVENTHOUSE_DATABASE: [
        {
            "question": "Are there any high-risk compliance alerts?",
            "query": """compliance_alerts\n| where risk_score > 80 and suppressed == false\n| project advisor_id, alert_type, risk_score, action_taken"""
        },
        {
            "question": "Show failed pre-trade compliance checks.",
            "query": """trading_events\n| where pre_trade_check == 'failed'\n| project advisor_id, client_id, instrument, action, timestamp\n| order by timestamp desc"""
        },
        {
            "question": "What core banking errors occurred today?",
            "query": """core_banking_clicks\n| where error_code != '' and timestamp > ago(24h)\n| summarize errors = count() by module, error_code\n| order by errors desc"""
        },
        {
            "question": "Show advisor branch presence.",
            "query": """branch_presence\n| where timestamp > ago(8h)\n| summarize latest = max(timestamp) by advisor_id, branch_id, location_type\n| order by latest desc"""
        },
        {
            "question": "What client interactions happened today?",
            "query": """client_interactions\n| where timestamp > ago(24h)\n| summarize calls = count(), avg_sec = avg(duration_sec) by channel, outcome\n| order by calls desc"""
        },
        {
            "question": "Are there incomplete post-trade filings?",
            "query": """trading_events\n| where post_trade_filing_status != 'completed'\n| where timestamp > ago(24h)\n| project trade_id, advisor_id, instrument, post_trade_filing_status"""
        },
        {
            "question": "Show compliance alert trends this week.",
            "query": """compliance_alerts\n| where timestamp > ago(7d)\n| summarize count() by bin(timestamp, 1d), alert_type\n| order by timestamp asc"""
        },
        {
            "question": "What are the most active trading instruments?",
            "query": """trading_events\n| where timestamp > ago(24h)\n| summarize trades = count() by instrument, action\n| order by trades desc"""
        },
    ],
}

print(f"OPS_FEWSHOTS set: {', '.join(f'{k}: {len(v)} queries' for k, v in OPS_FEWSHOTS.items())}")

## Sample Questions for the Finance Data Agents

### QA Agent — `Finance_QA_Agent`

| # | Category | Sample Question |
|---|----------|------------------|
| 1 | Advisors | Which advisors manage the highest AUM? |
| 2 | Advisors | Show advisors by branch and role. |
| 3 | Compliance | What is the average compliance document completion time by type? |
| 4 | Compliance | Which document types have the highest error rates? |
| 5 | Exams | Which branches are least ready for regulatory exams? |
| 6 | Exams | How many exam readiness gaps exist? |
| 7 | Clients | Show me client satisfaction by advisor. |
| 8 | Clients | Which clients have the most complaints? |
| 9 | Loans | What is the average loan processing time per stage? |
| 10 | Loans | What are the most common loan rejection reasons? |
| 11 | Quality | Which advisors have the most document quality issues? |
| 12 | Quality | Show document completeness scores by type. |
| 13 | KYC | How many KYC verifications failed per advisor? |
| 14 | KYC | What is the average KYC completion time? |
| 15 | Alerts | What are the most common regulatory alert types? |
| 16 | Trading | Show trade compliance documentation times. |
| 17 | Wellness | Which advisors have the highest documentation burden? |
| 18 | Branches | Show branch presence patterns for advisors. |
| 19 | Escalations | What is the case escalation rate by advisor? |
| 20 | Products | Show products by risk tier. |

### Ops Agent — `Finance_Ops_Agent`

| # | Category | Sample Question |
|---|----------|------------------|
| 1 | Burnout | Which advisors are at burnout risk? |
| 2 | Burnout | Who has the most overtime hours? |
| 3 | Exams | Which branches are not ready for upcoming exams? |
| 4 | Compliance | What compliance alerts need immediate attention? |
| 5 | Compliance | Are there high-risk compliance alerts in real-time? |
| 6 | Loans | Which loans are stuck in processing? |
| 7 | Trading | Show failed pre-trade compliance checks. |
| 8 | Trading | Are there incomplete post-trade filings? |
| 9 | KYC | Which KYC verifications failed? |
| 10 | Quality | Show document quality issues. |
| 11 | Alerts | What regulatory alerts need attention? |
| 12 | System | What core banking errors occurred recently? |
| 13 | Clients | Show clients with low NPS scores. |
| 14 | Escalations | What case escalations happened recently? |
| 15 | Presence | Which advisors are currently in meetings? |
| 16 | Trends | Show compliance alert trends this week. |

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# SAVE INSTRUCTIONS FOR PARENT NOTEBOOK
# ════════════════════════════════════════════════════════════════════════

import json

_result = json.dumps({
    "qa": QA_AGENT_INSTRUCTIONS,
    "ops": OPS_AGENT_INSTRUCTIONS,
    "qa_fewshots": QA_FEWSHOTS,
    "ops_fewshots": OPS_FEWSHOTS,
    "qa_ds_instructions": QA_DS_INSTRUCTIONS,
    "ops_ds_instructions": OPS_DS_INSTRUCTIONS
})

_tmp_path = "file:/lakehouse/default/Files/_temp/agent_instructions.json"
notebookutils.fs.put(_tmp_path, _result, True)
print(f"  Written {len(_result):,} bytes to {_tmp_path}")

print(f"{'═'*60}")
print(f"AGENT INSTRUCTIONS LOADED — {INDUSTRY.upper()}")
print(f"{'═'*60}")
print(f"")
print(f"  QA_AGENT_INSTRUCTIONS:  {len(QA_AGENT_INSTRUCTIONS):,} chars")
print(f"  OPS_AGENT_INSTRUCTIONS: {len(OPS_AGENT_INSTRUCTIONS):,} chars")
print(f"  QA_FEWSHOTS:  {sum(len(v) for v in QA_FEWSHOTS.values())} total queries across {len(QA_FEWSHOTS)} datasources")
print(f"  OPS_FEWSHOTS: {sum(len(v) for v in OPS_FEWSHOTS.values())} total queries across {len(OPS_FEWSHOTS)} datasources")

notebookutils.notebook.exit("ok")